In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# El notebook funciona desde la raíz o desde la carpeta notebooks.
ROOT = Path.cwd()
if not (ROOT / 'MASTER_CHART_F1000Research.xlsx').exists():
    ROOT = ROOT.parent
DATASET = ROOT / 'MASTER_CHART_F1000Research.xlsx'
ARTIFACTS = ROOT / 'artifacts'
print(f'Carpeta de trabajo: {ROOT}')

Carpeta de trabajo: c:\1-Universidad\6to\Analitica de datos\Johan\Momento3


In [2]:
raw = pd.read_excel(DATASET)
print(f'Registros: {raw.shape[0]} | Columnas: {raw.shape[1]}')
display(raw.head())
display(pd.DataFrame({
    'tipo': raw.dtypes.astype(str),
    'nulos': raw.isna().sum(),
    'unicos': raw.nunique(dropna=False)
}))

Registros: 303 | Columnas: 25


,GROUPS: DENGUE FEVER OR COMPLICATED DENGUE,AGE,GENDER,DURATION OF FEVER,HEADACHE,MYALGIA,ABDOMINAL PAIN,RASH,VOMITING,BREATHLESSNESS,...,LOWEST WBC,HIGHEST WBC,PLATELET COUNT,ALT,CREATININE,FERRITIN,IV FLUIDS,ANTIBIOTICS,ANTIPYRETICS,STEROIDS
0,2,2,1,3,2,1,2,2,1,2,...,3400,5100,30000,53.0,0.62,4167.0,1,1,2,1
1,2,2,1,4,2,1,2,2,2,2,...,3000,5300,43000,21.0,0.43,108.0,1,2,2,2
2,1,1,2,3,2,1,2,2,2,2,...,2300,2600,129000,12.0,0.34,212.0,1,1,1,2
3,1,2,2,5,2,1,2,2,2,2,...,1500,4800,70000,171.0,0.80,1429.0,1,1,1,1
4,1,2,2,3,2,1,2,2,2,2,...,8600,9100,184000,45.0,0.44,NaN,1,2,1,2


,tipo,nulos,unicos
GROUPS: DENGUE FEVER OR COMPLICATED DENGUE,int64,0,2
AGE,int64,0,2
GENDER,int64,0,2
DURATION OF FEVER,int64,0,10
HEADACHE,int64,0,2
MYALGIA,int64,0,2
ABDOMINAL PAIN,int64,0,2
RASH,int64,0,2
VOMITING,int64,0,2
BREATHLESSNESS,int64,0,2


In [3]:
# Distribución del objetivo antes de transformar códigos.
target_counts = raw.iloc[:, 0].value_counts().sort_index()
display(target_counts.rename(index={1: 'No complicado', 2: 'Complicado'}).to_frame('pacientes'))
print('Columnas constantes:', [c for c in raw.columns if raw[c].nunique(dropna=False) <= 1])

,pacientes
GROUPS: DENGUE FEVER OR COMPLICATED DENGUE,
No complicado,240
Complicado,63


Columnas constantes: []


In [4]:
import sys
sys.path.insert(0, str(ROOT))
from data_pipeline import MODEL_FEATURES, prepare_training_frame

X, y, constant_columns = prepare_training_frame(raw)
print('Columnas constantes eliminadas:', constant_columns or 'Ninguna')
print(f'Matriz final: {X.shape[0]} filas x {X.shape[1]} variables')
display(X.head())
display(X.isna().sum().sort_values(ascending=False).head(10).to_frame('nulos'))

Columnas constantes eliminadas: Ninguna
Matriz final: 303 filas x 29 variables


,age_over_6,gender_female,fever_days,headache,myalgia,abdominal_pain,rash,vomiting,breathlessness,bleeding,...,ferritin,iv_fluids,antibiotics,antipyretics,steroids,ferritin_missing,platelet_wbc_ratio,hb_pcv_ratio,symptom_count,treatment_count
0,1,0,3,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,4167.0,1.0,1.0,0.0,1.0,0.0,8.823529,0.308696,2.0,3.0
1,1,0,4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,108.0,1.0,0.0,0.0,0.0,0.0,14.333333,0.331606,2.0,1.0
2,0,1,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,212.0,1.0,1.0,1.0,0.0,0.0,56.086957,0.329577,2.0,3.0
3,1,1,5,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1429.0,1.0,1.0,1.0,1.0,0.0,46.666667,0.334917,2.0,4.0
4,1,1,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,NaN,1.0,0.0,1.0,0.0,1.0,21.395349,0.324390,2.0,2.0


,nulos
ferritin,112
alt,1
creatinine,1
gender_female,0
age_over_6,0
abdominal_pain,0
rash,0
vomiting,0
breathlessness,0
bleeding,0


In [5]:
display(pd.DataFrame({
    'grupo': ['Dengue no complicado', 'Dengue complicado'],
    'codigo_modelo': [0, 1],
    'pacientes': [(y == 0).sum(), (y == 1).sum()]
}))
display(X.describe().T[['mean', 'min', 'max']].round(2))

,grupo,codigo_modelo,pacientes
0,Dengue no complicado,0,240
1,Dengue complicado,1,63


,mean,min,max
age_over_6,0.80,0.00,1.0
gender_female,0.47,0.00,1.0
fever_days,3.98,1.00,10.0
headache,0.30,0.00,1.0
myalgia,0.30,0.00,1.0
abdominal_pain,0.17,0.00,1.0
rash,0.05,0.00,1.0
vomiting,0.42,0.00,1.0
breathlessness,0.01,0.00,1.0
bleeding,0.02,0.00,1.0


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

def make_pipeline(classifier):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', classifier),
    ])

logistic = make_pipeline(LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
neural = make_pipeline(MLPClassifier(
    hidden_layer_sizes=(24, 12), activation='relu', solver='adam',
    alpha=0.001, learning_rate_init=0.003, max_iter=1500,
    early_stopping=True, validation_fraction=0.2, n_iter_no_change=35,
    random_state=42
))

logistic.fit(X_train, y_train)
neural.fit(X_train, y_train)
print('Entrenamiento finalizado.')
print('Iteraciones de la red:', neural.named_steps['classifier'].n_iter_)

Entrenamiento finalizado.
Iteraciones de la red: 56


c:\1-Universidad\6to\Analitica de datos\Johan\Momento3\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


In [7]:
def evaluate(model, name):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    return {
        'modelo': name,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, prob),
        'matriz_confusion': confusion_matrix(y_test, pred).tolist(),
    }

results = pd.DataFrame([evaluate(logistic, 'Regresión logística'), evaluate(neural, 'Red neuronal')])
display(results.style.format({c: '{:.3f}' for c in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']}))

,modelo,accuracy,precision,recall,f1,roc_auc,matriz_confusion
0,Regresión logística,0.836,0.571,0.923,0.706,0.960,"[[39, 9], [1, 12]]"
1,Red neuronal,0.902,0.889,0.615,0.727,0.901,"[[47, 1], [5, 8]]"


In [8]:
# Variables con mayor peso absoluto en la regresión logística.
coefficients = pd.Series(
    logistic.named_steps['classifier'].coef_[0], index=MODEL_FEATURES
).sort_values(key=np.abs, ascending=False)
display(coefficients.head(10).to_frame('coeficiente'))
print('Coeficiente positivo: aumenta la probabilidad estimada del grupo complicado.')

,coeficiente
iv_fluids,0.814349
alt,0.792072
ferritin_missing,-0.776422
low_blood_pressure,-0.724340
platelet_wbc_ratio,-0.598399
rash,0.382840
treatment_count,0.377806
creatinine,-0.301726
age_over_6,-0.245808
breathlessness,0.191063


Coeficiente positivo: aumenta la probabilidad estimada del grupo complicado.


In [9]:
ARTIFACTS.mkdir(exist_ok=True)
def metric_dict(model):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    matrix = confusion_matrix(y_test, pred, labels=[0, 1])
    return {
        'accuracy': round(float(accuracy_score(y_test, pred)), 4),
        'precision': round(float(precision_score(y_test, pred, zero_division=0)), 4),
        'recall': round(float(recall_score(y_test, pred, zero_division=0)), 4),
        'f1': round(float(f1_score(y_test, pred, zero_division=0)), 4),
        'roc_auc': round(float(roc_auc_score(y_test, prob)), 4),
        'confusion_matrix': matrix.tolist(),
    }

metadata = {}
for column in ['hb', 'pcv', 'lowest_wbc', 'highest_wbc', 'platelet_count', 'alt', 'creatinine', 'ferritin']:
    values = X[column].dropna()
    metadata[column] = {'min': float(values.min()), 'max': float(values.max()), 'median': float(values.median())}

bundle = {
    'logistic': logistic, 'neural_network': neural,
    'feature_names': MODEL_FEATURES,
    'metrics': {'logistic': metric_dict(logistic), 'neural_network': metric_dict(neural)},
    'metadata': metadata,
    'dataset': {'rows': len(raw), 'raw_columns': raw.shape[1],
                'constant_columns_removed': constant_columns,
                'target_distribution': {'uncomplicated': int((y == 0).sum()), 'complicated': int((y == 1).sum())}},
}
joblib.dump(bundle, ARTIFACTS / 'models.joblib')
print(f"Bundle exportado en: {ARTIFACTS / 'models.joblib'}")

Bundle exportado en: c:\1-Universidad\6to\Analitica de datos\Johan\Momento3\artifacts\models.joblib
